In [2]:
# Core imports
import numpy as np
import torch
import sys
from pathlib import Path
# Add project root to Python path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

# Import modules
from src.data.dataset import create_data_loaders
from src.models.loganomaly import LogAnomalyModel
from src.engine.trainer import LogSeqTrainer
from src.utils.metrics import evaluate_model, print_metrics, save_experiment_results
from src.utils.data_loader import create_train_val_test_split, filter_normal_samples, load_loghub
from src.utils.visualizer import UniversalAnomalyVisualizer

In [2]:
# Define paths
DATA_DIR = '../data/bgl/preprocessed'

In [3]:
X, y, vocab = load_loghub(DATA_DIR)

vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")
print(f"Events: {sorted(vocab.keys())[:10]}")

INFO:src.utils.data_loader:Loading data from LogHub preprocessing: ../data/bgl/preprocessed
INFO:src.utils.data_loader:Loaded data:
INFO:src.utils.data_loader:  - Sequences: (949589, 20)
INFO:src.utils.data_loader:  - Labels: (949589,)
INFO:src.utils.data_loader:  - Normal: 868556, Anomaly: 81033
INFO:src.utils.data_loader:  - Loaded vocabulary: 371 events


Vocab size: 371
Events: ['<PAD>', '<UNK>', 'E1', 'E10', 'E100', 'E101', 'E102', 'E103', 'E1035', 'E1036']


In [4]:
# Split data (70/15/15)
splits = create_train_val_test_split(X, y, train_ratio=0.7, val_ratio=0.15, random_state=42)
(X_train, y_train), (X_val, y_val), (X_test, y_test) = splits['train'], splits['val'], splits['test']

# Filter to keep only normal samples for semi-supervised training
X_train, y_train = filter_normal_samples(X_train, y_train, verbose=True)

INFO:src.utils.data_loader:Splitting data: train=0.7, val=0.15, test=0.15
INFO:src.utils.data_loader:Split complete:
INFO:src.utils.data_loader:  - Train: 664711 samples (56723 anomalies)
INFO:src.utils.data_loader:  - Val:   142439 samples (12155 anomalies)
INFO:src.utils.data_loader:  - Test:  142439 samples (12155 anomalies)
INFO:src.utils.data_loader:======================================================================
INFO:src.utils.data_loader:FILTERING TRAINING DATA FOR SEMI-SUPERVISED LEARNING
INFO:src.utils.data_loader:======================================================================
INFO:src.utils.data_loader:Original training size: 664711 samples
INFO:src.utils.data_loader:  Normal samples: 607,988 (91.47%)
INFO:src.utils.data_loader:  Anomaly samples: 56,723 (8.53%)
INFO:src.utils.data_loader:
Filtered training size: 607,988 samples (NORMAL ONLY)
INFO:src.utils.data_loader:Removed 56,723 anomalies from training set
INFO:src.utils.data_loader:✓ Training data is now pur

In [5]:
# Create data loaders
batch_size = 64
train_loader, val_loader, test_loader = create_data_loaders(
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    batch_size=batch_size
)

In [6]:
#  Load the pre-computed embeddings
semantic_vectors = torch.load(f'{DATA_DIR}/semantic_embeddings.pt')
emb_dim = semantic_vectors.shape[1] 
print(f"Loaded semantic embeddings with dim: {emb_dim}")

Loaded semantic embeddings with dim: 384


In [7]:
# Create LogAnomaly model
model = LogAnomalyModel(
    vocab_size=vocab_size,
    embedding_dim=emb_dim,  
    hidden_dim=128,
    num_layers=2,
    dropout=0.3,
    use_attention=True, 
    n_heads=4, 
    semantic_embeddings=semantic_vectors 
)

In [8]:
# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Architecture:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Embedding dimension: {emb_dim}")
print(f"  Attention enabled: {model.use_attention}")
print(f"  Semantic embeddings: {'✓ Loaded' if semantic_vectors is not None else '✗ Random init'}")


Model Architecture:
  Total parameters: 651,891
  Trainable parameters: 651,891
  Embedding dimension: 384
  Attention enabled: True
  Semantic embeddings: ✓ Loaded


In [9]:
# Train
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
learning_rate = 0.001
patience = 20

print(f"Training on device: {device}")

trainer = LogSeqTrainer(model, device=device, learning_rate=learning_rate)

history = trainer.fit(
    train_loader, val_loader,
    num_epochs=50,
    early_stopping_patience=patience,
    print_every=5  # Print every 5 epochs
)

Training on device: mps


Training: 100%|██████████| 9500/9500 [00:58<00:00, 161.37it/s]



Epoch 1/50 - 71.77s
  Train Loss: 0.1957
  Val Loss:   0.7969
  ✓ New best model (val_loss: 0.7969)


Training: 100%|██████████| 9500/9500 [00:53<00:00, 176.42it/s]



Epoch 5/50 - 66.18s
  Train Loss: 0.1637
  Val Loss:   0.7764
  No improvement (1/20)


...


Epoch 30/50 - 67.09s
  Train Loss: 0.1612
  Val Loss:   0.7373
  No improvement (13/20)


Training: 100%|██████████| 9500/9500 [00:53<00:00, 176.95it/s]



Epoch 35/50 - 67.48s
  Train Loss: 0.1611
  Val Loss:   0.7943
  No improvement (18/20)


Training: 100%|██████████| 9500/9500 [00:53<00:00, 176.85it/s]



Early stopping triggered after 37 epochs

✓ Loaded best model (val_loss: 0.7358)
Total training time: 2480.46s


In [18]:
# Evaluate
# Capture predictions, true labels, AND anomaly scores from the loader
top_k = 11
predictions, true_labels, anomaly_scores = trainer.detect_anomalies(
    test_loader,
    top_k=top_k,
    return_scores=True
)

metrics = evaluate_model(predictions, true_labels) 
print_metrics(metrics)

Detecting anomalies: 100%|██████████| 2226/2226 [00:18<00:00, 118.42it/s]



EVALUATION METRICS
Accuracy:  0.9888 (98.88%)
Precision: 0.8885
Recall:    0.9936
F1-Score:  0.9381

Confusion Matrix:
              Predicted
              Normal  Anomaly
Actual Normal   128768     1516
       Anomaly      78    12077


In [19]:
# Save experiment results
save_experiment_results(
    filepath="../results/bgl_loganomaly_results.json",
    dataset="BGL",
    model_name="LogAnomaly",
    device=device,
    model=model,
    history=history,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
    max_len=int(np.percentile([len(s) for s in X_train], 95)),
    metrics=metrics,
    batch_size=batch_size,
    learning_rate=learning_rate,
    patience=patience,
    top_k=top_k,
    # LogAnomaly-specific params
    use_attention=True,
    n_heads=4
)

✓ Results saved to ../results/bgl_loganomaly_results.json


{'dataset': 'BGL',
 'model': 'LogAnomaly',
 'device': 'mps',
 'architecture': {'vocab_size': 371,
  'embedding_dim': 384,
  'hidden_dim': 128,
  'num_layers': 2,
  'dropout': 0.3,
  'use_attention': True,
  'num_attention_heads': 4},
 'training': {'num_epochs': 37,
  'batch_size': 64,
  'learning_rate': 0.001,
  'early_stopping_patience': 20,
  'best_val_loss': 0.7357535055049453,
  'training_time_seconds': 2480.4554221630096},
 'data': {'train_size': 607988,
  'val_size': 142439,
  'test_size': 142439,
  'max_sequence_length': 20,
  'train_normal_only': True},
 'detection': {'top_k': 11, 'method': 'next_event_prediction'},
 'metrics': {'accuracy': 0.9888092446591172,
  'precision': 0.8884720076509969,
  'recall': 0.9935828877005347,
  'f1': 0.9380922790119621,
  'confusion_matrix': [[128768, 1516], [78, 12077]]}}